# CLOY Processing

This notebook is for you to work with the CLOY data folder.

## 1. Turning every CLOY thread into a vector of words

### A. Loading all threads

First, I want to explore looking into one thread in the CLOY file.

In [26]:
import json

with open("RedditExplorer/cloy_chronological.jsonl") as f:
    first_line = f.readline()

thread = json.loads(first_line)

print(thread.keys())
type(thread)



dict_keys(['post_id', 'timestamp', 'author', 'subreddit', 'title', 'selftext', 'url', 'score', 'permalink', 'comments'])


dict

I dump all threads into a list for simplicity later

In [42]:
threads = []

with open("RedditExplorer/cloy_chronological.jsonl") as f:
    for line in f:
        threads.append(json.loads(line))
type (threads)

list

I look into a complete thread.

In [44]:
threads[4]

{'post_id': 'fj5byj',
 'timestamp': 1584294837,
 'author': 'titania73',
 'subreddit': 'CrashLandingOnYou',
 'title': 'Why I think we all need a second season of CLOY. (Spoilers)',
 'selftext': 'I binged crash landing on you this week and can’t stop thinking about this show! I love the characters, the story line, and how unique it is compared to a lot of other dramedies I’ve watched. But I have to admit that I was a bit disappointed with the ending. This seems to be an unpopular opinion as a lot of fans online say that it ended well and a second season may ruin it. They found their way back to each other and that’s what matters. \n\nI disagree. I can’t believe that this is the best future we get for these two characters. Seeing each other two weeks a year may be fine for now, but how is that a fair relationship for either of them long term? So much happens in a year - how can two weeks without any other communication be enough? We see time and time again how much they mean to each other

It seems like comments are under `comment_body`, and under each comment there is also `replies`. More comments are under `comment_body` under `replies` under comments.

### B. Getting all comments

This function keeps going until it reaches a comment that does not have `replies`. This function also gets all the comments nested under one comment.

In [45]:
def get_comment_sentence (comment):
    sentence = comment["comment_body"]
    if "replies" in comment:
        for reply in comment["replies"]:
            sentence = sentence + " "+get_comment_sentence(reply)

    return sentence


### C. Turn one entire thread into one string

The question asks for main post, that means I also need to extract the `title` and `selftext`.

In [46]:
def get_thread_sentence(thread):
    sentence = thread["title"] + " " + thread["selftext"]

    for comment in thread["comments"]:
        sentence += " " + get_comment_sentence(comment)

    return sentence


Let's see if this works on the first thread.

In [47]:
print(get_thread_sentence(thread[1]))

ok, let’s get this started! What are your thoughts about the show? I haven’t seen a kdrama for the longest time and it was actually my dad who chose to stream CLOY on Netflix. He stopped watching after the second episode but I went on after immediately falling in love with the leads. Hyun Bin is just so charismatic as Capt. Ri Jeong Hyeok and Son Ye-jin is perfect as Yoon Se-ri. She’s definitely got the rom-com and drama genre down pat! I also loved Capt. Ri’s squad, most especially Pyo Chi Su ☺️ (we can probably talk about individual characters next time).

CLOY tackled politics, drama and deception, family issues, friendships - and best of all, pure unconditional love, love brought about by fate - all in one highly entertaining and well-written package. 

I’m planning on doing a rewatch soon, when I get a chance! i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my

### D. Turn the string into words using textblob

In [48]:
from textblob import TextBlob

I tokenize each thread into words.

In [55]:
thread_words = []
for thread in threads:
    text = get_thread_sentence(thread)
    tblob = TextBlob(text)
    thread_words.append(tblob.words)


In [56]:
thread_words[0]

WordList(['Crash', 'Landing', 'On', 'You', 'has', 'been', 'created', 'deleted'])

I realize that the current code makes my words case sensitive. Therefore I lowercase the texts.

In [57]:
thread_words = []
for thread in threads:
    text = get_thread_sentence(thread)
    tblob = TextBlob(text.lower())
    thread_words.append(tblob.words)

In [58]:
thread_words[0]

WordList(['crash', 'landing', 'on', 'you', 'has', 'been', 'created', 'deleted'])

### E. Create the vocabulary of all unique terms

I am a bit stuck here. I don't really know how to create the vacabulary, and I am not sure how to capture unique terms. I googled and learned about `set`

In [59]:
vocab = set()
for words in thread_words:
    for word in words:
        vocab.add(word)

In [62]:
vocab = sorted(list(vocab))
print(vocab[:20])

["'-i", "'-sshi", "'d", "'ll", "'m", "'re", "'s", "'ve", "'🫠", '0', '0.0000001', '0.0001', '0.001', '0.1', '0.30', '0.5', '007', '00:10', '00:11', '00:30']


### F. Create vector with the frequency

I count how frequent each vocabulary word appears in each thread.

In [66]:
vectors = []
for words in thread_words:
    vector = []
    for word in vocab:
        vector.append(words.count(word))
    vectors.append(vector)
        


KeyboardInterrupt: 

It has been 20 minutes and the code is still running, let's check the size of data.

In [77]:
print("number of threads:", len(thread_words))
print("number of vvocabs:", len(vocab))
print("number of words in the first thread:", len(thread_words[0]))

number of threads: 3075
number of vvocabs: 36445
number of words in the first thread: 8


Visualize the frequency vector with a df.

In [65]:
import pandas as pd

df = pd.DataFrame(vectors, columns=vocab)

df.head()

NameError: name 'vectors' is not defined